### Importing libraries and Loading data

In [8]:
# Run this BEFORE Cell 1
# Tells us exactly what's wrong!

import sys
print(f"1. Python: {sys.executable}")

try:
    import sentence_transformers
    print(f"2. sentence_transformers installed")
except:
    print(f"2. sentence_transformers MISSING!")

try:
    import faiss
    print(f"3. faiss installed")
except:
    print(f"3. faiss MISSING!")

import os
cf = '../data/processed/chunks.json'
ff = '../vector_store/faiss_index/alzheimer.index'
print(f"4. chunks.json: {'✅' if os.path.exists(cf) else '❌'}")
print(f"5. alzheimer.index: {'✅' if os.path.exists(ff) else '❌'}")

1. Python: c:\Users\tpriy\anaconda3\envs\semantic_agent\python.exe
2. sentence_transformers installed
3. faiss installed
4. chunks.json: ✅
5. alzheimer.index: ✅


In [9]:
# Load All Components ────────────
import faiss
import numpy as np
import json
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# ── Paths ──────────────────────────────────
CHUNKS_FILE = '../data/processed/chunks.json'
FAISS_FILE  = '../vector_store/faiss_index/alzheimer.index'

# ── Load components ────────────────────────
print(" Loading components...")

# Load chunks
with open(CHUNKS_FILE) as f:
    chunks = json.load(f)
print(f" Chunks loaded:    {len(chunks):,}")

# Load FAISS index
index = faiss.read_index(FAISS_FILE)
print(f" FAISS loaded:     {index.ntotal:,} vectors")

# Load model
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print(f" Model loaded:     PubMedBERT 768-dim")
print(f"\n All components ready!")

 Loading components...
 Chunks loaded:    40,995
 FAISS loaded:     40,995 vectors


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 561.69it/s, Materializing param=pooler.dense.weight]                               


 Model loaded:     PubMedBERT 768-dim

 All components ready!


In [10]:
#  Search Function:
def search_papers(query, k=3):
    """
    Search for relevant chunks
    using FAISS semantic search
    """
    # Encode query
    query_vector = model.encode([query])
    query_vector = query_vector.astype('float32')

    # Search FAISS
    distances, indices = index.search(
        query_vector, k=k
    )

    # Get results
    results = []
    for dist, idx in zip(
        distances[0], indices[0]
    ):
        chunk = chunks[idx]
        results.append({
            "title":    chunk['title'],
            "year":     chunk['year'],
            "authors":  chunk['authors'],
            "journal":  chunk['journal'],
            "chunk":    chunk['chunk'],
            "distance": float(dist)
        })

    return results

print("Search function ready!")

Search function ready!


### Test Query 1  

In [11]:
#  Test 1 — Biomarkers ────────────
query = "What blood biomarkers detect Alzheimer's early?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Authors:  {r['authors'][:40]}")
    print(f"   Journal:  {r['journal'][:40]}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

 Query: What blood biomarkers detect Alzheimer's early?

 Result 1:
   Title:    Evidence to Consider Angiotensin II Receptor Blockers for th...
   Year:     2016
   Authors:  Saavedra JM
   Journal:  Cellular and molecular neurobiology
   Distance: 99.31
   Preview:  in normal individuals when reliable biomarkers of Alzheimer's disease risk are identified....

 Result 2:
   Title:    Validating blood tests as a possible routine diagnostic assa...
   Year:     2023
   Authors:  Kodosaki E, Zetterberg H, Heslegrave A
   Journal:  Expert review of molecular diagnostics
   Distance: 111.12
   Preview:  In recent years, exciting developments in disease modifying treatments for Alzheimer's disease (AD) have made accurate and timely diagnosis of this di...

 Result 3:
   Title:    [How Helpful are Blood and Cerebrospinal Fluid Biomarkers fo...
   Year:     2023
   Authors:  Tokuda T
   Journal:  Brain and nerve = Shinkei kenkyu no shin
   Distance: 111.18
   Preview:  fluid biomarkers were a

### Test Query 2

In [12]:
#  Test 2 — Diabetes ──────────────
query = "How does diabetes connect to Alzheimer's disease?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

 Query: How does diabetes connect to Alzheimer's disease?

 Result 1:
   Title:    Can Alzheimer's Disease Be Secondary to Type-2 Diabetes Mell...
   Year:     2022
   Distance: 126.82
   Preview:  research revealed that insulin impacts various neurotransmitters in a way that enhances memory and cognition. Thirdly, several pathological research h...

 Result 2:
   Title:    [Diabetes type 2 and Alzheimer disease - one or two diseases...
   Year:     2013
   Distance: 133.05
   Preview:  are similar for both diseases as well. The question is raised: can Alzheimer be a new form of diabetes disease?...

 Result 3:
   Title:    True or false? Alzheimer's disease is type 3 diabetes: Evide...
   Year:     2024
   Distance: 142.38
   Preview:  Globally, Alzheimer's disease (AD) is the most widespread chronic neurodegenerative disorder, leading to cognitive impairment, such as aphasia and agn...


###  Test Query 3

In [13]:
# Test 3 — Treatment ─────────────
query = "What is Lecanemab and how does it treat Alzheimer's?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

 Query: What is Lecanemab and how does it treat Alzheimer's?

 Result 1:
   Title:    The Black Book of Psychotropic Dosing and Monitoring....
   Year:     2024
   Distance: 138.14
   Preview:  decades, these drugs modestly improve cognition in Alzheimer's disease patients and do not alter the progressive course of the illness. Lecanemab is a...

 Result 2:
   Title:    The Black Book of Psychotropic Dosing and Monitoring....
   Year:     2024
   Distance: 142.99
   Preview:  number of patients who progress to severe Alzheimer's disease and require institutional care. The standard dose is 10 mg/kg given via IV over one hour...

 Result 3:
   Title:    Reconsidering repurposing: long-term metformin treatment imp...
   Year:     2024
   Distance: 144.33
   Preview:  when this drug is used for individuals with AD....


### Search Quality Report

In [14]:
# Search Quality Report ──────────
print(f"\n {'='*55}")
print(f"SEARCH QUALITY REPORT")
print(f"{'='*55}")

test_queries = [
    "blood biomarkers Alzheimer's detection",
    "diabetes insulin resistance brain",
    "Lecanemab treatment amyloid",
    "APOE4 genetic risk dementia",
    "deep learning MRI Alzheimer's"
]

for query in test_queries:
    results = search_papers(query, k=1)
    best    = results[0]
    print(f"\n {query[:45]}...")
    print(f"   Best match: {best['title'][:50]}...")
    print(f"   Year:       {best['year']}")
    print(f"   Distance:   {best['distance']:.2f}")

print(f"\n{'='*55}")
print(f" Search pipeline working!")
print(f"   Index:   {index.ntotal:,} vectors")
print(f"   Model:   PubMedBERT 768-dim")
print(f"   Status:  READY FOR RAG! 🚀")
print(f"{'='*55}")




SEARCH QUALITY REPORT

 blood biomarkers Alzheimer's detection...
   Best match: Plasma biomarkers for Alzheimer's and related deme...
   Year:       2024
   Distance:   89.66

 diabetes insulin resistance brain...
   Best match: State of the Science on Brain Insulin Resistance a...
   Year:       2024
   Distance:   82.97

 Lecanemab treatment amyloid...
   Best match: Lecanemab: A Second in Class Therapy for the Manag...
   Year:       2024
   Distance:   96.06

 APOE4 genetic risk dementia...
   Best match: Leading determinants of incident dementia among in...
   Year:       2024
   Distance:   88.88

 deep learning MRI Alzheimer's...
   Best match: Advanced MRI based Alzheimer's diagnosis through e...
   Year:       2025
   Distance:   78.92

 Search pipeline working!
   Index:   40,995 vectors
   Model:   PubMedBERT 768-dim
   Status:  READY FOR RAG! 🚀
